In [1]:
import sys
import os

os.chdir("/home/duoc/workflow")
sys.path.insert(0, "/home/duoc/workflow") 

In [2]:
# Cell 1 fix
from ddgs import DDGS
from app.rag.loader import DocumentLoader

loader = DocumentLoader()

query = "Giá iPhone 16 Việt Nam 2025"
with DDGS() as ddgs:
    results = list(ddgs.text(query, max_results=5))

docs = []
for r in results:
    try:
        doc = loader.load_web(r["href"])
        docs.append(doc)
        print(f"✅ {r['href'][:60]} — {len(doc.text)} chars")
    except Exception as e:
        print(f"❌ {r['href'][:60]} — {e}")

print(f"\nTổng: {len(docs)} trang")

✅ https://vi.wikipedia.org/wiki/IPhone_16_Pro — 6607 chars
❌ https://www.facebook.com/groups/1368303459846390/posts/27562 — Không trích xuất được nội dung từ URL: https://www.facebook.com/groups/1368303459846390/posts/27562910349959010/
❌ https://www.facebook.com/VatVoStudio69/posts/so-sánh-iphone- — Không trích xuất được nội dung từ URL: https://www.facebook.com/VatVoStudio69/posts/so-sánh-iphone-17-iphone-16-iphone-15-iphone-14-lên-đời-sao-cho-đáng-giávideo-nà/1372696164653668/
✅ https://iphonequangngai.com/ — 264 chars
✅ https://tuannguyenmobile.com/thu-mua-iphone-cu/?srsltid=AfmB — 8836 chars

Tổng: 3 trang


In [8]:
import asyncio
from crawl4ai import AsyncWebCrawler

async def crawl(url: str):
    async with AsyncWebCrawler() as crawler:
        result = await crawler.arun(url)
        print(f"✅ {url[:60]}")
        print(f"   {len(result.markdown)} chars")
        print(result.markdown[:500])
        return result.markdown

url = "https://www.google.com/search?q=laptop gaming dưới 20 triệu 2025"
result = await crawl(url)

[INIT].... → Crawl4AI 0.8.6 

[FETCH]... ↓ https://www.google.com/search?q=laptop gaming dưới 20 triệu 2025                                     |
✓ | ⏱: 1.27s 

[SCRAPE].. ◆ https://www.google.com/search?q=laptop gaming dưới 20 triệu 2025                                     |
✓ | ⏱: 0.08s 

[COMPLETE] ● https://www.google.com/search?q=laptop gaming dưới 20 triệu 2025                                     |
✓ | ⏱: 1.39s 

✅ https://www.google.com/search?q=laptop gaming dưới 20 triệu 
   37120 chars
Bỏ qua để đến phần nội dung chính[Hỗ trợ truy cập](https://support.google.com/websearch/answer/181196?hl=vi)
[](https://www.google.com/webhp?hl=vi&sa=X&ved=0ahUKEwiWg4mi4eOUAxUUslYBHZMWH_EQPAgI "Đến trang chủ Google")
[](https://www.google.com/webhp?hl=vi&ictx=0&sa=X&ved=0ahUKEwiWg4mi4eOUAxUUslYBHZMWH_EQpYkNCAo)
Nhấn / để chuyển tới hộp tìm kiếm
laptop gaming dưới 20 triệu 2025
![](https://www.google.com/tia/tia.png)
[](https://www.google.com.vn/intl/vi/about/products?tab=wh)
[Đăng nhập](https:/


In [3]:
from playwright.async_api import async_playwright
from ddgs import DDGS

async def search_and_browse(query: str, max_results: int = 3):
    # Bước 1: lấy URLs từ DDGS
    with DDGS() as ddgs:
        results = list(ddgs.text(query, max_results=max_results))
    
    print(f"🔍 Tìm thấy {len(results)} URLs")
    
    # Bước 2: browse từng URL
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        pages_content = []
        
        for r in results:
            url = r["href"]
            try:
                page = await browser.new_page(user_agent=(
                    "Mozilla/5.0 (X11; Linux x86_64) "
                    "AppleWebKit/537.36 (KHTML, like Gecko) "
                    "Chrome/120.0.0.0 Safari/537.36"
                ))
                await page.goto(url, timeout=15000)
                await page.wait_for_load_state("domcontentloaded", timeout=10000)
                text = await page.inner_text("body")
                pages_content.append({"url": url, "text": text})
                print(f"✅ {url[:60]} — {len(text)} chars")
                print(text[:200])
                print("---")
                await page.close()
            except Exception as e:
                print(f"❌ {url[:60]} — {e}")
        
        await browser.close()
        return pages_content

result = await search_and_browse("laptop gaming dưới 20 triệu 2025")

🔍 Tìm thấy 3 URLs
✅ https://www.youtube.com/watch?v=UcVFa0K6Th8 — 11 chars
0:00 / 6:13
---
✅ https://thinkpro.vn/noi-dung/lam-viec-tai-gia-giai-tri-tha-g — 9254 chars
1900.63.3579
Hỗ trợ
Địa chỉ cửa hàng
Tin tức
Sản phẩm
Đăng nhập
Tin tức
Top 3 laptop gaming dưới 25 triệu đáng mua năm 2025
Thu Hồng
16:50 25-10-2024

Thấu hiểu nhu cầu của người dùng yêu công nghệ đa
---
✅ https://blog.mygear.vn/laptop-gaming-tu-15-den-20-trieu/ — 7221 chars
Chuyển đến nội dung
TRANG CHỦ
TIN CÔNG NGHỆ
ĐÁNH GIÁ
GAME
REVIEW – VIDEO
THỦ THUẬT
TƯ VẤN
TỔNG HỢP

Trang chủ » Tin công nghệ » Tư vấn chọn cấu hình laptop gaming từ 15 đến 20 triệu phù hợp

CHỦ ĐỀ BL
---


In [13]:
import nodriver as uc
from bs4 import BeautifulSoup

async def search_google(query: str):
    browser = await uc.start()
    page = await browser.get(f"https://www.google.com/search?q={query}")
    await page.wait(2)
    html = await page.get_content()
    browser.stop()
    
    soup = BeautifulSoup(html, "html.parser")
    # Lấy search results
    results = []
    for g in soup.select("div.g"):
        title = g.select_one("h3")
        link = g.select_one("a")
        snippet = g.select_one(".VwiC3b")
        if title and link:
            results.append({
                "title": title.text,
                "href": link.get("href"),
                "snippet": snippet.text if snippet else ""
            })
    return results

results = await search_google("laptop gaming dưới 20 triệu 2025")
for r in results:
    print(f"📌 {r['title']}")
    print(f"   {r['href']}")
    print(f"   {r['snippet'][:100]}")
    print()

ConnectionClosedError: no close frame received or sent

In [9]:
import subprocess
subprocess.Popen(["Xvfb", ":99", "-screen", "0", "1920x1080x24"])
import os
os.environ["DISPLAY"] = ":99"
import nodriver as uc
from bs4 import BeautifulSoup

async def search_google(query: str, max_results: int = 2):
    browser = await uc.start(headless=False)
    page = await browser.get(f"https://www.google.com/search?q={query}")
    await page.wait(3)
    html = await page.get_content()
    browser.stop()
    
    soup = BeautifulSoup(html, "html.parser")
    seen = set()
    links = []
    skip = ("google", "youtube", "facebook", "twitter", "instagram")
    
    for a in soup.find_all("a", href=True):
        href = a["href"]
        text = a.get_text(strip=True)
        if (
            href.startswith("http")
            and not any(s in href for s in skip)
            and href not in seen
            and text
        ):
            seen.add(href)
            links.append({"href": href, "text": text})
        if len(links) >= max_results:
            break
    
    for l in links:
        print(f"📌 {l['text'][:60]}")
        print(f"   {l['href'][:80]}")
    
    return links

links = await search_google("laptop gaming dưới 20 triệu 2025", max_results=10)

The XKEYBOARD keymap compiler (xkbcomp) reports:
> Warning:          Could not resolve keysym XF86CameraAccessEnable
> Warning:          Could not resolve keysym XF86CameraAccessDisable
> Warning:          Could not resolve keysym XF86CameraAccessToggle
> Warning:          Could not resolve keysym XF86NextElement
> Warning:          Could not resolve keysym XF86PreviousElement
> Warning:          Could not resolve keysym XF86AutopilotEngageToggle
> Warning:          Could not resolve keysym XF86MarkWaypoint
> Warning:          Could not resolve keysym XF86Sos
> Warning:          Could not resolve keysym XF86NavChart
> Warning:          Could not resolve keysym XF86FishingChart
> Warning:          Could not resolve keysym XF86SingleRangeRadar
> Warning:          Could not resolve keysym XF86DualRangeRadar
> Warning:          Could not resolve keysym XF86RadarOverlay
> Warning:          Could not resolve keysym XF86TraditionalSonar
> Warning:          Could not resolve keysym XF86Clearvu

📌 Laptop Gaming dưới 20 triệu chính hãng, hiệu năng tốt ...GEA
   https://gearvn.com/collections/laptop-gaming-gia-duoi-20-trieu?srsltid=AfmBOopV0
📌 25+ laptop Gaming dưới 20 triệu pin trâu, hiệu năng cao ...Đ
   https://dienthoaivui.com.vn/tin-tuc/laptop-gaming-duoi-20-trieu
📌 Top 10 Laptop Cho Sinh Viên Dưới 20 Triệu Đáng Mua ...An Phá
   https://www.anphatpc.com.vn/top-10-laptop-cho-sinh-vien-duoi-20-trieu-dang-mua-n
📌 Laptop dưới 20 triệu | Thiết kế sang trọng, mạnh mẽ, bền bỉC
   https://cellphones.com.vn/bo-loc/laptop-tu-15-20-trieu
📌 5 mẫu laptop gaming dưới 25 triệu 2025 nổi bật, cấu hình ...
   https://fptshop.com.vn/tin-tuc/danh-gia/laptop-gaming-duoi-25-trieu-2025-175437
📌 Top laptop gaming dưới 20 triệu đáng mua nhất 2025Phong Vũht
   https://phongvu.vn/cong-nghe/top-laptop-gaming-duoi-20-trieu-dang-mua/?srsltid=A
📌 Top ASUS Laptop Gaming Dưới 20 Triệu Chính HãngASUShttps://v
   https://vn.store.asus.com/eshop-mua-asus-top-laptop-gaming-duoi-20-trieu-manh-nh
📌 Laptop dưới 2

The XKEYBOARD keymap compiler (xkbcomp) reports:
> Warning:          Could not resolve keysym XF86CameraAccessEnable
> Warning:          Could not resolve keysym XF86CameraAccessDisable
> Warning:          Could not resolve keysym XF86CameraAccessToggle
> Warning:          Could not resolve keysym XF86NextElement
> Warning:          Could not resolve keysym XF86PreviousElement
> Warning:          Could not resolve keysym XF86AutopilotEngageToggle
> Warning:          Could not resolve keysym XF86MarkWaypoint
> Warning:          Could not resolve keysym XF86Sos
> Warning:          Could not resolve keysym XF86NavChart
> Warning:          Could not resolve keysym XF86FishingChart
> Warning:          Could not resolve keysym XF86SingleRangeRadar
> Warning:          Could not resolve keysym XF86DualRangeRadar
> Warning:          Could not resolve keysym XF86RadarOverlay
> Warning:          Could not resolve keysym XF86TraditionalSonar
> Warning:          Could not resolve keysym XF86Clearvu

In [2]:
import asyncio
import logging
from app.rag.crawl import ResearchService

logging.basicConfig(level=logging.INFO)

async def main():
    service = ResearchService(
        headless=True,
        browser_timeout=5,
        max_browser_retries=2,
    )
    
    result = await service.run("python async programming", max_results=3)
    print(f"Found {result['metadata']['successful_crawls']} documents")

await main()

ModuleNotFoundError: No module named 'app.rag.crawl'

In [8]:
import asyncio
from ddgs import DDGS
from crawl4ai import AsyncWebCrawler

async def rag_search(query: str, max_results: int = 3):
    print(f"🔍 Query: {query}")

    # 1. Search URLs
    with DDGS() as ddgs:
        results = list(ddgs.text(query, max_results=max_results))

    urls = [r["href"] for r in results]

    print("\n📌 URLs:")
    for i, url in enumerate(urls, 1):
        print(f"{i}. {url}")

    # 2. Crawl URLs
    async with AsyncWebCrawler() as crawler:
        crawl_results = await crawler.arun_many(urls=urls)

    # 3. Build RAG docs
    docs = []

    print("\n📚 Documents:\n")

    for result in crawl_results:
        if result.success:
            text = result.markdown[:5000]  # giới hạn để test

            docs.append(
                {
                    "url": result.url,
                    "content": text,
                }
            )

            print(f"✅ {result.url}")
            print(f"   {len(text)} chars")
            print(text[:300].replace("\n", " "))
            print("-" * 80)

        else:
            print(f"❌ {result.url}")
            print(result.error_message)

    return docs


docs = await rag_search(
    "laptop gaming dưới 20 triệu 2025",
    max_results=10
)

print(f"\n🎯 Tổng số documents: {len(docs)}")

🔍 Query: laptop gaming dưới 20 triệu 2025

📌 URLs:
1. https://vitinhnguyenthang.com/laptop-gaming-duoi-25-trieu-dang-mua-nhat
2. https://tinhte.vn/thread/co-nen-mua-laptop-gaming-duoi-20-trieu-dong-nhung-dieu-can-biet-truoc-khi-quyet-dinh.4032747/
3. https://minhkhoa.com.vn/laptop-gaming-duoi-20-trieu/
4. https://laptopk1.vn/so-sanh-lenovo-loq-2024-va-loq-2025
5. https://huynhlongstore.com/laptop-gaming-tai-vinh-long-cap-nhat-mau-moi-nhat-2024-2025/
6. https://taphoaonl.com/6-mau-laptop-asus-duoi-20-trieu-2025-hieu-nang-manh-thiet-ke-dep/
7. https://fptshop.com.vn/tin-tuc/danh-gia/laptop-duoi-10-trieu-dang-mua-nhat-207122
8. https://www.themost10.com/top-laptop-gaming-duoi-20-trieu/
9. https://thuthuattonghop.com/tags/laptop-gaming-rtx-4060-dang-mua/
10. https://www.gamergear.vn/top-6-mau-laptop-gaming-duoi-25-trieu-dang-mua-nhat-2023/


[INIT].... → Crawl4AI 0.8.6 

[FETCH]... ↓ https://taphoaonl.com/6-mau-laptop-asus-duoi-20-trieu-2025-hieu-nang-manh-thiet-ke-dep/              |
✓ | ⏱: 1.81s 

[SCRAPE].. ◆ https://taphoaonl.com/6-mau-laptop-asus-duoi-20-trieu-2025-hieu-nang-manh-thiet-ke-dep/              |
✓ | ⏱: 0.02s 

[COMPLETE] ● https://taphoaonl.com/6-mau-laptop-asus-duoi-20-trieu-2025-hieu-nang-manh-thiet-ke-dep/              |
✓ | ⏱: 1.85s 

...

[COMPLETE] ● https://huynhlongstore.com/laptop-gaming-tai-vinh-long-cap-nhat-mau-moi-nhat-2024-2025/              |
✓ | ⏱: 5.85s 

[FETCH]... ↓ https://thuthuattonghop.com/tags/laptop-gaming-rtx-4060-dang-mua/                                    |
✓ | ⏱: 6.73s 

[SCRAPE].. ◆ https://thuthuattonghop.com/tags/laptop-gaming-rtx-4060-dang-mua/                                    |
✓ | ⏱: 0.02s 

[COMPLETE] ● https://thuthuattonghop.com/tags/laptop-gaming-rtx-4060-dang-mua/                                    |
✓ | ⏱: 6.77s 


📚 Documents:

✅ https://taphoaonl.com/6-mau-laptop-asus-duoi-20-trieu-2025-hieu-nang-manh-thiet-ke-dep/
   1039 chars
Please enable cookies.  #  Error 1033 Ray ID: a046d2bd7faa6e43 • 2026-05-31 14:56:05 UTC ##  Cloudflare Tunnel error  ##  What happened?  You've requested a page on a website (taphoaonl.com) that is on the [Cloudflare](https://www.cloudflare.com/5xx-error-landing/) network. The host (taphoaonl.com) 
--------------------------------------------------------------------------------
✅ https://www.gamergear.vn/top-6-mau-laptop-gaming-duoi-25-trieu-dang-mua-nhat-2023/
   5000 chars
[Skip to content](https://www.gamergear.vn/top-6-mau-laptop-gaming-duoi-25-trieu-dang-mua-nhat-2023/#main) [ ![Gamer Gear Store](https://www.gamergear.vn/wp-content/uploads/2023/01/logo-34-1400x387.png)![Gamer Gear Store](https://www.gamergear.vn/wp-content/uploads/2023/01/logo-34-1400x387.png)](htt
--------------------------------------------------------------------------------
✅ https://vitinhng

In [1]:
import sys
import os

os.chdir("/home/duoc/workflow")
sys.path.insert(0, "/home/duoc/workflow") 


from app.tools.research_tool import ResearchTool


async def main():
    tool = ResearchTool()  # Khởi tạo instance
    result = await tool.run("laptop gaming dưới 20 triệu 2025", n_results=5, n_crawl=3)
    
    lines = [
        f"Query: {result['query']}\n",
        "Sources:",
    ]
    for src in result["sources"]:
        lines.append(f"  - {src}")
    
    if result["content"]:
        lines.extend(["\nContent:", result["content"][:10000]])
    
    return "\n".join(lines)

await main()

[INIT].... → Crawl4AI 0.8.6 

[FETCH]... ↓ https://dienthoaivui.com.vn/tin-tuc/laptop-gaming-duoi-20-trieu                                      |
✓ | ⏱: 1.21s 

[SCRAPE].. ◆ https://dienthoaivui.com.vn/tin-tuc/laptop-gaming-duoi-20-trieu                                      |
✓ | ⏱: 0.19s 

[COMPLETE] ● https://dienthoaivui.com.vn/tin-tuc/laptop-gaming-duoi-20-trieu                                      |
✓ | ⏱: 1.45s 

[FETCH]... ↓ https://gearvn.com/collections/laptop-gaming-gia...mgxIrMfrgBV3Iq8SkLPatoIT2xXDauUDQFF6FLJ1vWDLJyTc  |
✓ | ⏱: 1.87s 

...

[COMPLETE] ● https://gearvn.com/collections/laptop-gaming-gia...mgxIrMfrgBV3Iq8SkLPatoIT2xXDauUDQFF6FLJ1vWDLJyTc  |
✓ | ⏱: 2.15s 

[FETCH]... ↓ https://phongvu.vn/cong-nghe/top-laptop-gaming-d...P5tNyT018vYEfzEV0T4qhEtlX2q9qruu21lsqNzlxcLElmRz  |
✓ | ⏱: 2.17s 

[SCRAPE].. ◆ https://phongvu.vn/cong-nghe/top-laptop-gaming-d...P5tNyT018vYEfzEV0T4qhEtlX2q9qruu21lsqNzlxcLElmRz  |
✓ | ⏱: 0.06s 

[COMPLETE] ● https://phongvu.vn/cong-nghe/top-laptop-gaming-d...P5tNyT018vYEfzEV0T4qhEtlX2q9qruu21lsqNzlxcLElmRz  |
✓ | ⏱: 2.26s 

'Query: laptop gaming dưới 20 triệu 2025\n\nSources:\n  - 1. Laptop Gaming dưới 20 triệu chính hãng, hiệu năng tốt ...GEARVNhttps://gearvn.com› laptop-gaming-gia-duoi-20-trieu — https://gearvn.com/collections/laptop-gaming-gia-duoi-20-trieu?srsltid=AfmBOoo-mgxIrMfrgBV3Iq8SkLPatoIT2xXDauUDQFF6FLJ1vWDLJyTc\n  - 2. 25+ laptop Gaming dưới 20 triệu pin trâu, hiệu năng cao ...Điện Thoại Vuihttps://dienthoaivui.com.vn› ... › Review Công nghệ — https://dienthoaivui.com.vn/tin-tuc/laptop-gaming-duoi-20-trieu\n  - 3. Top laptop gaming dưới 20 triệu đáng mua nhất 2025Phong Vũhttps://phongvu.vn› Home › Thủ thuật — https://phongvu.vn/cong-nghe/top-laptop-gaming-duoi-20-trieu-dang-mua/?srsltid=AfmBOooXP5tNyT018vYEfzEV0T4qhEtlX2q9qruu21lsqNzlxcLElmRz\n  - 4. Tiêu chí lựa chọn laptop gaming — https://phongvu.vn/cong-nghe/top-laptop-gaming-duoi-20-trieu-dang-mua/#I_Tieu_chi_lua_chon_laptop_gaming\n  - 5. II. Top 3 laptop gaming dưới... — https://phongvu.vn/cong-nghe/top-laptop-gaming-duoi-20-trieu-dang